In [0]:
%run ./secrets

In [0]:
import requests

# def get_data(path, ver="/v1"):
#     headers = {'user-agent': 'my-agent/1.0.1',
#                'password': get_proxy_password(),
#                'Accept': 'application/json'}
#     full_url = f"{PROXY_BASE_URL}{ver}{API_BASE_PREFIX}{path}"
#     response = requests.get(full_url, headers=headers)
#     return response

def get_headers():
    return {'user-agent': 'my-agent/1.0.1',
            'password': get_proxy_password(),
            'Accept': 'application/json'}

def get_data(endpoint, path_params=None, query_params=None):
    headers = get_headers()
    path_template = ENDPOINTS.get(endpoint)
    if path_params:
        full_path = path_template.format(**path_params)
    else:
        full_path = path_template
    full_url = f"{PROXY_BASE_URL}{API_VERSION}{full_path}"
    # fill query params
    response = requests.get(full_url, headers=headers, params=query_params)
    return response

def get_flights(origin, destination, date, query_params=None):
    headers = get_headers()
    path_template = ENDPOINTS.get(EndpointKeys.FLIGHTSTATUS_BY_ROUTE)
    full_path = path_template.format(origin=origin, destination=destination, date=date)
    full_url = f"{PROXY_BASE_URL}{API_VERSION}{full_path}"
    response = requests.get(full_url, headers=headers, params=query_params)
    return response

def fetch_data(endpoint, path_params=None, query_params=None, max_retries=3, timeout=20):
    headers = get_headers()
    path_template = ENDPOINTS.get(endpoint)

    if not path_template:
        raise ValueError(f"Unknown endpoint: {endpoint}")

    if path_params:
        full_path = path_template.format(**path_params)
    else:
        full_path = path_template
    full_url = f"{PROXY_BASE_URL}{API_VERSION}{full_path}"

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(
                full_url,
                headers=headers,
                params=query_params,
                timeout=timeout,
            )

            if response.status_code in {429, 500, 502, 503, 504}:
                if attempt == max_retries:
                    response.raise_for_status()
                time.sleep(2 ** (attempt - 1))
                continue

            response.raise_for_status()
            return response

        except (requests.Timeout, requests.ConnectionError):
            if attempt == max_retries:
                raise
            time.sleep(2 ** (attempt - 1))